In [ ]:
%load_ext autoreload
%autoreload 2

# 03 — EDA on the synthetic data

Does the simulator behave as the data card says? Simulated values only (loans, statements, defaults).

In [1]:
import numpy as np
import pandas as pd

from mkopoguard import config
from mkopoguard.data.checks import failed_checks, plausibility_report
from mkopoguard.viz import apply_style, histogram, paired_bars, ranked_bars, save

apply_style()
IMG = config.PROJECT_ROOT / "docs" / "img"
applicants = pd.read_parquet(config.DATA_PROCESSED / "applicants.parquet")
transactions = pd.read_parquet(config.DATA_PROCESSED / "transactions.parquet")
names = (
    applicants.drop_duplicates("country_code")
    .set_index("country_code")["country"]
    .astype(str)
    .to_dict()
)
names["CIV"] = "Côte d'Ivoire"

report = plausibility_report(applicants, transactions)
print("Failed checks:", failed_checks(report) or "none")
report.round(3)

Failed checks: none


default_rate                             0.199
default_rate_kenya                       0.161
country_default_rate_min                 0.155
country_default_rate_max                 0.274
median_loan_kes                      11000.000
median_transactions_per_applicant      216.500
max_transactions_in_one_day             18.000
median_final_balance_months              0.556
p99_final_balance_months                 3.456
share_using_overdraft                    0.230
max_overdraft_owed_months                1.000
share_without_income                     0.000
dtype: float64

In [2]:
by_country = applicants.groupby("country_code")["defaulted"].mean()
fig = ranked_bars(
    by_country,
    "Default rates range from 16% to 27% by country",
    "Simulated defaults; country is never used by the default rule",
    labels=names,
)
save(fig, IMG / "eda_06_default_rate_by_country.png")

WindowsPath('D:/MkopoGuard/docs/img/eda_06_default_rate_by_country.png')

**Takeaway:** default rates vary from 16% (Gabon, Kenya) to 27% (Zimbabwe) even though the default rule never uses country. The differences come from each country's real mix of income, employment and savings habits.

In [3]:
table = applicants.groupby("income_quintile")["defaulted"].mean().to_frame("Default rate")
table.index = table.index.astype(int)
fig = paired_bars(
    table,
    "Poorer applicants default more often",
    "Simulated default rate by income quintile (1 = poorest 20%)",
    xlabel="Income quintile",
)
save(fig, IMG / "eda_07_default_rate_by_income.png")
table.round(3)

,Default rate
income_quintile,
1,0.251
2,0.228
3,0.200
4,0.185
5,0.161


**Takeaway:** default falls steadily from 25% in the poorest quintile to 16% in the richest, as the data card intends (through income pattern, stress and loan-to-income, not income directly).

In [4]:
fig = histogram(
    {"Loans": applicants["loan_amount_kes"]},
    "Most loans are between KES 5,000 and 25,000",
    "Simulated loan amounts, share of applicants (log scale)",
    "Loan amount (KES)",
    bins=np.geomspace(1_000, 100_000, 16),
    log_x=True,
)
save(fig, IMG / "eda_08_loan_amounts.png")
applicants["loan_amount_kes"].describe().round(0)

count     18718.0
mean      18037.0
std       19652.0
min        1000.0
25%        5000.0
50%       11000.0
75%       23500.0
max      100000.0
Name: loan_amount_kes, dtype: float64

**Takeaway:** median loan KES 11,000, about one month's income. Bins are wide on purpose: loans are rounded to KES 500, so narrow bins at the low end would show empty gaps. 229 loans sit at the KES 100,000 cap.

In [5]:
per_applicant = transactions.groupby("applicant_id").size()
fig = histogram(
    {"Statements": per_applicant},
    "A typical statement has about 220 transactions",
    "Transactions per applicant over 180 days (9,848 applicants with mobile money)",
    "Transactions in the statement",
    bins=np.arange(0, 760, 20),
)
save(fig, IMG / "eda_09_transactions_per_applicant.png")
pattern = applicants.set_index("applicant_id").loc[per_applicant.index, "hidden_income_pattern"]
per_applicant.groupby(pattern, observed=True).median()

hidden_income_pattern
irregular    273.0
salaried     139.0
seasonal     103.0
dtype: float64

**Takeaway:** two humps. Irregular earners get 8–20 small payments a month, each followed by a cash withdrawal, so their statements are about twice as long (median 273) as salaried (139) and seasonal (103) earners. The number of inflows will be a useful feature.

In [6]:
income = applicants.set_index("applicant_id")["hidden_monthly_income_kes"]


def final_balance_months(tx):
    final = tx.groupby("applicant_id")["balance_after"].last()
    return final / income.loc[final.index]


before = pd.read_parquet(config.DATA_INTERIM / "transactions_step16.parquet")
fig = histogram(
    {
        "Before fix (step 1.6)": final_balance_months(before),
        "After fix (step 2.2)": final_balance_months(transactions),
    },
    "Adding cash withdrawals stopped balances piling up",
    "Balance at the end of the statement, in months of income",
    "Final balance (months of income)",
    bins=np.arange(0, 8.25, 0.25),
)
save(fig, IMG / "eda_10_final_balance_before_after.png")
print("Old data fails:", failed_checks(plausibility_report(applicants, before)))

Old data fails: {'median_final_balance_months': '4.323 is outside 0.1 to 2.0', 'p99_final_balance_months': '7.681 is outside 0.5 to 6.0', 'max_overdraft_owed_months': '11.000 is outside 0.0 to 1.0'}


**Takeaway:** the step 1.6 simulator spent only about half of each applicant's income, so the median final balance was 4.3 months of income and overdraft debt reached 11.6 months. Adding cash withdrawals and a one-month overdraft limit brought these to 0.56 and 1.0. The plausibility checks now stop this from coming back.